# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ruchitgoud/flyrankai-intern/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os

repo_path = "/content/flyrankai-intern"

if not os.path.exists(repo_path):
    !git clone https://github.com/ruchitgoud/flyrankai-intern.git

%cd /content/flyrankai-intern

print("Current folder:", os.getcwd())

/content/flyrankai-intern
Current folder: /content/flyrankai-intern


In [2]:
!ls data/raw

content_refresh_anonymized.csv


In [3]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))
print(df.columns.tolist())

Rows: 30000
Columns: 44
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [4]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## My Rule

I will create a simple refresh-priority score using two observable signals:

1. **Content staleness** — pages that have not been updated for a long time receive more priority.
2. **CTR relative to search position** — pages with reasonable search visibility but relatively low CTR receive additional priority.

The score will rank pages for human review. It will not automatically decide that a page must be refreshed.

### Reason codes

- `STALE_CONTENT` — the page has a high number of days since its last update.
- `LOW_CTR_FOR_POSITION` — the page has search visibility and position but relatively low CTR.
- `STALE_AND_LOW_CTR` — both signals indicate a potential refresh opportunity.
- `GENERAL_REVIEW` — neither primary signal is strong enough to assign a specific reason.

### Action

The rule uses `REVIEW_REFRESH` for higher-priority items and `MONITOR` for lower-priority items. These labels support human review and do not automatically determine the correct action.

### Signal 1 — Content staleness

I checked content age using `days_since_last_update`.

The data shows that the 0–30 day bucket has an average trend of +0.78%, while the 91–180 day bucket has an average trend of -15.68%.

This supports the idea that stale content can be useful for prioritizing refresh review.

**Verdict: CONFIRMED, with a caveat.** The 181+ day bucket has only 174 rows, so its result should be treated cautiously. This signal is directional rather than proof that updating a page will improve performance.

In [5]:
# Signal 1 — Content staleness

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 30, 90, 180, float("inf")],
    labels=["0-30 days", "31-90 days", "91-180 days", "181+ days"]
)

staleness_table = (
    df.groupby("staleness_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          avg_trend_pct=("trend_pct", "mean"),
          avg_ctr=("ctr", "mean")
      )
      .reset_index()
)

display(staleness_table)

,staleness_bucket,n,avg_trend_pct,avg_ctr
0,0-30 days,20480,0.784405,0.609021
1,31-90 days,175,-7.373054,0.117543
2,91-180 days,9171,-15.683224,0.238367
3,181+ days,174,-6.781203,3.693276


### Signal 2 — CTR relative to search position

I checked CTR across search-position buckets.

CTR decreases from 2.71% for positions 1–3 to 0.21% for positions 21+, showing that CTR is strongly related to search position.

However, the trend is not consistent across the position buckets. Positions 4–10 and 11–20 have negative average trends, while positions 1–3 and 21+ have positive average trends.

**Verdict: MIXED.** CTR should therefore be interpreted relative to search position rather than treated as a standalone refresh signal.

In [6]:
# Signal 2 — CTR vs search position

df["position_bucket"] = pd.cut(
    df["avg_position"],
    bins=[0, 3, 10, 20, float("inf")],
    labels=["1-3", "4-10", "11-20", "21+"]
)

ctr_position_table = (
    df.groupby("position_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          avg_ctr=("ctr", "mean"),
          avg_trend_pct=("trend_pct", "mean")
      )
      .reset_index()
)

display(ctr_position_table)

,position_bucket,n,avg_ctr,avg_trend_pct
0,1-3,1141,2.714303,8.255130
1,4-10,11842,0.651045,-12.491247
2,11-20,7273,0.323443,-10.491449
3,21+,8539,0.211333,9.173176


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [7]:
import os
import numpy as np

baseline = df.copy()

baseline["staleness_score"] = (
    baseline["days_since_last_update"]
    .rank(pct=True)
    .fillna(0)
)

baseline["position_bucket"] = pd.cut(
    baseline["avg_position"],
    bins=[0, 3, 10, 20, float("inf")],
    labels=["1-3", "4-10", "11-20", "21+"]
)

position_median_ctr = (
    baseline.groupby("position_bucket", observed=False)["ctr"]
    .transform("median")
)

baseline["ctr_gap"] = (
    position_median_ctr - baseline["ctr"]
).clip(lower=0)

baseline["ctr_gap_score"] = (
    baseline["ctr_gap"]
    .rank(pct=True)
    .fillna(0)
)


baseline["score"] = (
    0.60 * baseline["staleness_score"]
    + 0.40 * baseline["ctr_gap_score"]
) * 100

baseline["reason_code"] = np.select(
    [
        (baseline["staleness_score"] >= 0.75) &
        (baseline["ctr_gap_score"] >= 0.75),

        baseline["staleness_score"] >= 0.75,

        baseline["ctr_gap_score"] >= 0.75
    ],
    [
        "STALE_AND_LOW_CTR",
        "STALE_CONTENT",
        "LOW_CTR_FOR_POSITION"
    ],
    default="GENERAL_REVIEW"
)

baseline["action"] = np.where(
    baseline["score"] >= 50,
    "REVIEW_REFRESH",
    "MONITOR"
)

baseline = baseline.sort_values(
    "score",
    ascending=False
).reset_index(drop=True)

baseline["rank"] = baseline.index + 1

output = baseline[
    [
        "rank",
        "content_id",
        "score",
        "reason_code",
        "action",
        "days_since_last_update",
        "ctr",
        "avg_position"
    ]
].copy()

os.makedirs("work/outputs", exist_ok=True)

output.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Ranked queue created.")
print("Rows:", len(output))
print("Top 10:")
display(output.head(10))

print("\nSaved to:")
print("work/outputs/baseline_action_score.csv")

Ranked queue created.
Rows: 30000
Top 10:


,rank,content_id,score,reason_code,action,days_since_last_update,ctr,avg_position
0,1,content_55a5b1c46474,97.188484,STALE_AND_LOW_CTR,REVIEW_REFRESH,373,0.0,7.5
1,2,content_1b4ec72dafd4,97.183484,STALE_AND_LOW_CTR,REVIEW_REFRESH,372,0.0,7.0
2,3,content_06e19c6486b0,97.177484,STALE_AND_LOW_CTR,REVIEW_REFRESH,334,0.0,5.0
3,4,content_e2b702f4f92b,97.177484,STALE_AND_LOW_CTR,REVIEW_REFRESH,334,0.0,9.3
4,5,content_02b0d6e30129,97.171484,STALE_AND_LOW_CTR,REVIEW_REFRESH,313,0.0,6.9
5,6,content_f488400fca67,97.157484,STALE_AND_LOW_CTR,REVIEW_REFRESH,305,0.0,5.7
6,7,content_07ce98c6085a,97.141484,STALE_AND_LOW_CTR,REVIEW_REFRESH,304,0.0,5.3
7,8,content_7736e6144f3b,97.141484,STALE_AND_LOW_CTR,REVIEW_REFRESH,304,0.0,5.2
8,9,content_ab27c30d81f4,97.141484,STALE_AND_LOW_CTR,REVIEW_REFRESH,304,0.0,8.9
9,10,content_84d12054c0c0,97.141484,STALE_AND_LOW_CTR,REVIEW_REFRESH,304,0.0,8.0



Saved to:
work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [8]:
# Top-20 review

top20 = output.head(20).copy()

top20["confidence_note"] = np.where(
    (top20["days_since_last_update"] >= 180) &
    (top20["ctr"] == 0),
    "Strong baseline signal: very stale and zero CTR.",
    "Moderate signal: one or both baseline signals are weaker."
)

top20["what_would_make_it_wrong"] = np.where(
    top20["ctr"] == 0,
    "The page may have limited search opportunity or insufficient recent impressions.",
    "CTR may be reasonable for the page's search position."
)

display(
    top20[
        [
            "rank",
            "content_id",
            "action",
            "reason_code",
            "score",
            "confidence_note",
            "what_would_make_it_wrong"
        ]
    ]
)

,rank,content_id,action,reason_code,score,confidence_note,what_would_make_it_wrong
0,1,content_55a5b1c46474,REVIEW_REFRESH,STALE_AND_LOW_CTR,97.188484,Strong baseline signal: very stale and zero CTR.,The page may have limited search opportunity o...
1,2,content_1b4ec72dafd4,REVIEW_REFRESH,STALE_AND_LOW_CTR,97.183484,Strong baseline signal: very stale and zero CTR.,The page may have limited search opportunity o...
2,3,content_06e19c6486b0,REVIEW_REFRESH,STALE_AND_LOW_CTR,97.177484,Strong baseline signal: very stale and zero CTR.,The page may have limited search opportunity o...
3,4,content_e2b702f4f92b,REVIEW_REFRESH,STALE_AND_LOW_CTR,97.177484,Strong baseline signal: very stale and zero CTR.,The page may have limited search opportunity o...
4,5,content_02b0d6e30129,REVIEW_REFRESH,STALE_AND_LOW_CTR,97.171484,Strong baseline signal: very stale and zero CTR.,The page may have limited search opportunity o...
5,6,content_f488400fca67,REVIEW_REFRESH,STALE_AND_LOW_CTR,97.157484,Strong baseline signal: very stale and zero CTR.,The page may have limited search opportunity o...
6,7,content_07ce98c6085a,REVIEW_REFRESH,STALE_AND_LOW_CTR,97.141484,Strong baseline signal: very stale and zero CTR.,The page may have limited search opportunity o...
7,8,content_7736e6144f3b,REVIEW_REFRESH,STALE_AND_LOW_CTR,97.141484,Strong baseline signal: very stale and zero CTR.,The page may have limited search opportunity o...
8,9,content_ab27c30d81f4,REVIEW_REFRESH,STALE_AND_LOW_CTR,97.141484,Strong baseline signal: very stale and zero CTR.,The page may have limited search opportunity o...
9,10,content_84d12054c0c0,REVIEW_REFRESH,STALE_AND_LOW_CTR,97.141484,Strong baseline signal: very stale and zero CTR.,The page may have limited search opportunity o...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [9]:
# Section 4 — Weak picks + leakage check
print("Weakest picks in the top 20:")
display(
    top20[
        [
            "rank",
            "content_id",
            "score",
            "reason_code",
            "action",
            "confidence_note",
            "what_would_make_it_wrong"
        ]
    ].tail(5)
)

print("\nLeakage check:")

leakage_columns = [
    "trend_direction",
    "trend_pct",
    "health_score",
    "priority_score",
    "action_type"
]

present_leakage = [
    col for col in leakage_columns
    if col in baseline.columns
]

print("Potential label/product-decision columns present in dataframe:")
print(present_leakage)

print("\nBaseline score inputs:")
print([
    "days_since_last_update",
    "ctr",
    "avg_position"
])

print("\nNo trend_direction, trend_pct, health_score, priority_score, or action_type "
      "was used to calculate the baseline score.")

Weakest picks in the top 20:


,rank,content_id,score,reason_code,action,confidence_note,what_would_make_it_wrong
15,16,content_8bc10f396d2e,97.033484,STALE_AND_LOW_CTR,REVIEW_REFRESH,Strong baseline signal: very stale and zero CTR.,The page may have limited search opportunity o...
16,17,content_107776820988,97.033484,STALE_AND_LOW_CTR,REVIEW_REFRESH,Strong baseline signal: very stale and zero CTR.,The page may have limited search opportunity o...
17,18,content_3b7c76f80c79,97.033484,STALE_AND_LOW_CTR,REVIEW_REFRESH,Strong baseline signal: very stale and zero CTR.,The page may have limited search opportunity o...
18,19,content_b694314765e5,97.033484,STALE_AND_LOW_CTR,REVIEW_REFRESH,Strong baseline signal: very stale and zero CTR.,The page may have limited search opportunity o...
19,20,content_b84c5db2dbad,97.033484,STALE_AND_LOW_CTR,REVIEW_REFRESH,Strong baseline signal: very stale and zero CTR.,The page may have limited search opportunity o...



Leakage check:
Potential label/product-decision columns present in dataframe:
['trend_direction', 'trend_pct']

Baseline score inputs:
['days_since_last_update', 'ctr', 'avg_position']

No trend_direction, trend_pct, health_score, priority_score, or action_type was used to calculate the baseline score.


## Weak picks + leakage check

Some top-ranked pages may be weak recommendations if they have very low search opportunity or limited evidence behind their performance.

A page with zero CTR, for example, does not automatically mean that its content should be refreshed. It could have very little search demand or insufficient impressions.

The baseline score uses only observable inputs available in the dataset: content staleness and CTR relative to search position.

I deliberately did not use `trend_direction`, `trend_pct`, or existing product decision fields such as health or priority scores in the baseline calculation. This avoids using label-derived or product-decision information as inputs.

The baseline should therefore be treated as a directional decision-support rule, not proof that a page needs a refresh.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.